In [ ]:
import torch
torch.__version__

In [ ]:
# Set Device to cuda globally
if torch.cuda.is_available():
    torch.set_default_device('cuda')
else:
    torch.set_default_device('cpu')

In [ ]:
# AutoGrad and Backward

In [ ]:
x = torch.tensor(3., requires_grad=True)
y = torch.tensor(4., requires_grad=True)
z = 2 * x * y + 3

In [ ]:
z.backward()

In [ ]:
x.grad, y.grad

In [ ]:
z1 = torch.log(1 + torch.exp(-(x + y))) * (x + y)

In [ ]:
z1.backward()

In [ ]:
x.grad, y.grad

In [ ]:
# Training a NN and prediction

In [ ]:
# load the data

In [ ]:
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import v2
import matplotlib.pyplot as plt
import torch.nn.functional as F

In [ ]:
training_data = datasets.FashionMNIST(
    root = "data",
    train = True,
    download = True,
    transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    # target_transform=v2.Lambda(
    #     lambda y: F.one_hot(torch.tensor(y), num_classes=10).float())
)
testing_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = True,
    transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    # target_transform=v2.Lambda(
    #     lambda y: F.one_hot(torch.tensor(y), num_classes=10).float())
)

In [ ]:
# Prepare data for training

In [ ]:
cuda_generator = torch.Generator(device='cuda')

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True,
                              generator=cuda_generator)
test_dataloader = DataLoader(testing_data, batch_size=64, shuffle=True,
                             generator=cuda_generator)

In [ ]:
# Display image and label.
train_features, train_labels = next(iter(train_dataloader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")
img = train_features[0].squeeze()
label = train_labels[0]
plt.imshow(img, cmap="gray")
plt.show()
print(f"Label: {label}")

In [ ]:
#build NN

In [ ]:
from torch import nn

In [ ]:
class NN(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [ ]:
model = NN().to('cuda')
model

In [ ]:
X = torch.rand(1, 28, 28, generator=cuda_generator)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

In [ ]:
# Train the above network with traning data

In [ ]:
# Optimiztion with hyperparameters
learning_rate = 1e-3
batch_size = 64
epoch = 5

In [ ]:
# Optimization loop: We update the hyperparaters at each epoch
# Loss function
loss_fn = nn.CrossEntropyLoss()
# optimzier: Here it's Stochastic Gradient Descent
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        #Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
# Start the loops:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

In [ ]:
# Save the model

In [ ]:
import torchvision.models as models

In [ ]:
model = models.vgg16(weights='IMAGENET1K_V1')
torch.save(model.state_dict(), 'model_weights.pth')
# torch.save(model, 'model.pth')